# nb_01_bronze_to_silver - clean the raw files into typed Delta tables

Bronze is the CSV exactly as it arrived (`Files/bronze/history/`). Silver is the same data, but:

- **explicit schemas** (no guessing types from text),
- **casts** to timestamp / date / double,
- **trimmed, lower-cased** text codes,
- **deduplicated** on `order_id` (bronze deliberately has ~0.5% duplicate order rows).

Output tables: `silver_customers`, `silver_orders`, `silver_shipments`, `silver_daily_ops`, `silver_weather` (Delta, overwrite).

**Parameters.** Where the bronze files live, relative to the attached lakehouse.

In [ ]:
bronze_path = "Files/bronze/history"

**Imports and a helper.** `read_csv` applies a schema we declare, so a bad file fails loudly instead of silently becoming strings.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")   # order_ts values are UTC; keep them that way

def read_csv(name, schema):
    return (spark.read.option("header", True).option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss'Z'")
            .schema(schema).csv(f"{bronze_path}/{name}"))

def trim_lower(df, *cols):
    for c in cols:
        df = df.withColumn(c, F.lower(F.trim(F.col(c))))
    return df

def save(df, table):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"wrote {table}: {df.count()} rows")

**Explicit schemas** for the five files (kit 02 §4.2).

In [ ]:
customers_schema = StructType([
    StructField("customer_id", IntegerType()), StructField("region_id", IntegerType()),
    StructField("signup_date", DateType()), StructField("segment", StringType())])

orders_schema = StructType([
    StructField("order_id", IntegerType()), StructField("customer_id", IntegerType()),
    StructField("order_ts", TimestampType()), StructField("region_id", IntegerType()),
    StructField("warehouse_id", IntegerType()), StructField("carrier_id", IntegerType()),
    StructField("ship_method", StringType()), StructField("items_count", IntegerType()),
    StructField("total_weight_kg", DoubleType()), StructField("order_value", DoubleType()),
    StructField("promised_date", DateType()), StructField("channel", StringType())])

shipments_schema = StructType([
    StructField("order_id", IntegerType()), StructField("shipped_ts", TimestampType()),
    StructField("delivered_ts", TimestampType()), StructField("carrier_id", IntegerType())])

daily_ops_schema = StructType([
    StructField("date", DateType()), StructField("warehouse_id", IntegerType()),
    StructField("backlog_index", DoubleType()), StructField("staffing_level", StringType())])

weather_schema = StructType([
    StructField("date", DateType()), StructField("region_id", IntegerType()),
    StructField("forecast_severity", IntegerType()), StructField("actual_severity", IntegerType())])

**Orders: the table with duplicates.** Count before, drop duplicates on `order_id`, count after. This is the number learners should see change.

In [ ]:
orders_raw = read_csv("orders.csv", orders_schema)
orders_raw = trim_lower(orders_raw, "ship_method", "channel")

before = orders_raw.count()
orders = orders_raw.dropDuplicates(["order_id"])
after = orders.count()
print(f"orders: {before} rows before dedupe, {after} after, {before - after} duplicate rows removed")
save(orders, "silver_orders")

**Customers and shipments.** One row per customer / per order. Shipments carry the delivery time we use for the **label only** (never as a feature).

In [ ]:
customers = trim_lower(read_csv("customers.csv", customers_schema), "segment").dropDuplicates(["customer_id"])
save(customers, "silver_customers")

shipments = read_csv("shipments.csv", shipments_schema).dropDuplicates(["order_id"])
save(shipments, "silver_shipments")

**Daily ops and weather.** One row per day per warehouse / per region. `actual_severity` stays in silver but is forbidden as a feature (it's known only after delivery).

In [ ]:
daily_ops = trim_lower(read_csv("daily_ops.csv", daily_ops_schema), "staffing_level").dropDuplicates(["date", "warehouse_id"])
save(daily_ops, "silver_daily_ops")

weather = read_csv("weather_daily.csv", weather_schema).dropDuplicates(["date", "region_id"])
save(weather, "silver_weather")

**Check cell.** Null counts on the key columns should be zero, and every order should have exactly one shipment.

In [ ]:
o = spark.read.table("silver_orders"); s = spark.read.table("silver_shipments")
nulls = o.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["order_id", "order_ts", "carrier_id", "promised_date", "order_value"]])
nulls.show()
print("orders without a shipment:", o.join(s, "order_id", "left_anti").count())
print("order date range:", o.agg(F.min("order_ts"), F.max("order_ts")).collect()[0])